In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from collections import Counter, defaultdict
from itertools import compress
from scipy.io import loadmat
from scipy.optimize import linear_sum_assignment
from scipy.sparse.linalg import svds
from scipy.special import huber
from sklearn.preprocessing import StandardScaler
from time import process_time

import warnings
warnings.simplefilter('ignore')

In [ ]:
def segment_search(f, grad_f, x, y, tol=1e-6, stepsize=True):
    
    '''
    Minimizes f over [x, y], i.e., f(x+gamma*(y-x)) as a function of scalar gamma in [0,1]
    '''
    
    # restrict segment of search to [x, y]
    d = (y-x).copy()
    left, right = x.copy(), y.copy()
    
    # if the minimum is at an endpoint
    if np.dot(d, grad_f(x))*np.dot(d, grad_f(y)) >= 0:
        if f(y) <= f(x):
            return y, 1
        else:
            return x, 0
    
    # apply golden-section method to segment
    gold = (1+np.sqrt(5))/2
    improv = np.inf
    while improv > tol:
        old_left, old_right = left, right
        new = left+(right-left)/(1+gold)
        probe = new+(right-new)/2
        if f(probe) <= f(new):
            left, right = new, right
        else:
            left, right = left, probe
        improv = np.linalg.norm(f(right)-f(old_right))+np.linalg.norm(f(left)-f(old_left))
    x_min = (left+right)/2
    
    # compute step size gamma
    gamma = 0
    if stepsize == True:
        for i in range(len(d)):
            if d[i] != 0:
                gamma = (x_min[i]-x[i])/d[i]
                break
                
    return x_min, gamma

def sigd(f, grad_f, x, S, alpha):
    
    '''
    Simplex Gradient Descent
    '''
    
    g = np.dot(S, grad_f(x))
    e, k = np.ones(len(S)), len(S)
    d = g-np.dot(g, e)*e/k
    
    if np.linalg.norm(d) == 0:
        return S[0], [S[0]], [1]
    
    eta = np.min([alpha[i]/d[i] if d[i] > 0 else np.inf for i in range(len(alpha))])
    beta = np.array(alpha)-eta*d
    y = np.dot(beta, S)

    if f(x) >= f(y):
        idx = list(beta > 0)
        return y, list(compress(S, idx)), list(compress(beta, idx))
    
    else:
        x, gamma = segment_search(f, grad_f, x, y)
        return x, S, list((1-gamma)*np.array(alpha)+gamma*beta)

def find_index(v, S):
    
    for i in range(len(S)):
        if np.all(S[i] == v):
            return i
        
    return -1

In [ ]:
def fw(f, grad_f, L, x, step='ls', f_tol=1e-6, time_tol=np.inf):
    
    values, times, oracles, gaps = [f(x)], [0], [0], []
    f_improv = np.inf
    
    while f_improv > f_tol and np.sum(times) < time_tol:

        f_old = f(x)
        start = process_time()
        
        grad_f_x = grad_f(x)        
        v = lmo(grad_f_x)
        
        t1 = process_time()
        gaps.append(np.dot(grad_f_x, x-v))
        t2 = process_time()

        if step == 't':
            gamma = 2/(len(values)+1)
            x = (1-gamma)*x+gamma*v
        elif step == 'L':
            gamma = min(np.dot(grad_f_x, x-v)/(L*np.linalg.norm(x-v)**2), 1)
            x = (1-gamma)*x+gamma*v
        elif step == 'ls':
            x, gamma = segment_search(f, grad_f, x, v)
        else:
            d = v-x
            gamma = min(-np.dot(d, grad_f_x)/np.dot(d, (np.dot(step, d))), 1)
            x = x+gamma*d
        
        end = process_time()
        values.append(f(x))
        times.append(end-start+t1-t2)
        oracles.append(1)
        f_improv = f_old-f(x)
        
    return x, values, times, oracles, gaps

def afw(f, grad_f, L, x, step='ls', f_tol=1e-6, time_tol=np.inf):
    
    values, times, oracles, gaps = [f(x)], [0], [0], []
    f_improv = np.inf
    S, alpha = [x.copy()], [1]
    
    while f_improv > f_tol and np.sum(times) < time_tol:
        
        f_old = f(x)        
        start = process_time()
                
        grad_f_x = grad_f(x)
        v = lmo(grad_f_x)
        
        t1 = process_time()
        gaps.append(np.dot(grad_f_x, x-v))
        t2 = process_time()
        
        i_max = np.argmax(np.dot(S, grad_f_x))
        a = S[i_max]
        
        if np.dot(grad_f_x, v-x) <= np.dot(grad_f_x, x-a):
            if step == 'L':
                gamma = min(np.dot(grad_f_x, x-v)/(L*np.linalg.norm(x-v)**2), 1)
                x = (1-gamma)*x+gamma*v
            elif step == 'ls':
                x, gamma = segment_search(f, grad_f, x, v)
            else:
                d = v-x
                gamma = min(-np.dot(d, grad_f_x)/np.dot(d, (np.dot(step, d))), 1)
                x = x+gamma*d
            S.append(v)
            alpha = list((1-gamma)*np.array(alpha))+[gamma]
        
        else:
            gamma_max = alpha[i_max]/(1-min(alpha[i_max], 0.99999))
            if step == 'L':
                gamma = min(np.dot(grad_f_x, a-x)/(L*np.linalg.norm(a-x)**2), gamma_max)
                x = (1+gamma)*x-gamma*a
                alpha = list((1+gamma)*np.array(alpha))
                alpha[i_max] -= gamma
                if gamma == gamma_max:
                    del alpha[i_max]
                    del S[i_max]
            elif step == 'ls':
                x, gamma = segment_search(f, grad_f, x, (1+gamma_max)*x-gamma_max*a)
                alpha = list((1+gamma*gamma_max)*np.array(alpha))
                alpha[i_max] -= gamma*gamma_max
                if gamma == 1:
                    del alpha[i_max]
                    del S[i_max]
            else:
                d = x-a
                gamma = min(-np.dot(d, grad_f_x)/np.dot(d, (np.dot(step, d))), gamma_max)
                x = x+gamma*d
                alpha = list((1+gamma)*np.array(alpha))
                alpha[i_max] -= gamma
                if gamma == gamma_max:
                    del alpha[i_max]
                    del S[i_max]
                    
        end = process_time()
        values.append(f(x))
        times.append(end-start+t1-t2)
        oracles.append(2)
        f_improv = f_old-f(x)

    return x, values, times, oracles, gaps

In [ ]:
def align(d, hat_d):
    
    # If the zero vector is given for d hat, then return -1
    if np.linalg.norm(hat_d) < 1e-15:
        return -1
    
    else:
        return np.dot(d, hat_d)/(np.linalg.norm(d)*np.linalg.norm(hat_d))

def nnmp(x, grad_f_x, align_tol, K, traffic):
    
    d, Lbd, flag = np.zeros(len(x)), 0, True
    
    G = grad_f_x+d
    align_d = align(-grad_f_x, d)
    
    for k in range(K):
        
        if traffic == True:
            G = np.maximum(G, 0)
        
        u = lmo(G)-x
        d_norm = np.linalg.norm(d)
        if d_norm > 0 and np.dot(G, -d/d_norm) < np.dot(G, u):
            u = -d/d_norm
            flag = False
        lbd = -np.dot(G, u)/np.linalg.norm(u)**2
        dd = d+lbd*u
        align_dd = align(-grad_f_x, dd)
        align_improv = align_dd-align_d
        
        if align_improv > align_tol:
            d = dd
            Lbd = Lbd+lbd if flag == True else Lbd*(1-lbd/d_norm)
            G = grad_f_x+d
            align_d = align_dd
            flag = True
            
        else:
            break
        
    return d/Lbd, k, align_d

def boostfw(f, grad_f, L, x, step='ls', f_tol=1e-6, time_tol=np.inf, align_tol=1e-3, K=500, traffic=False):
    
    if traffic == False:
        values, times, oracles, gaps = [f(x)], [0], [0], [np.dot(grad_f(x), x-lmo(grad_f(x)))]
        f_improv = np.inf

        start = process_time()
        
        # Initialize x_0 as the lmo of some given initial point (called y in the paper)
        x = lmo(grad_f(x))
        end = process_time()
        values.append(f(x))
        times.append(end-start)
        oracles.append(1)
    else:
        values, times, oracles, gaps = [f(x)], [0], [0], []
        f_improv = np.inf
    
    while f_improv > f_tol and np.sum(times) < time_tol:
                
        f_old = f(x)
        start = process_time()
        
        grad_f_x = grad_f(x)
        
        t1 = process_time()
        gaps.append(np.dot(grad_f_x, x-lmo(grad_f_x)))
        t2 = process_time()
        
        # Here we compute step direction g and its alignment
        g, num_oracles, align_g = nnmp(x, grad_f_x, align_tol, K, traffic)
        
        # Depending on step size schedule chosen, compute the step size
        if step == 'L':
            gamma = min(align_g*np.linalg.norm(grad_f_x)/(L*np.linalg.norm(g)), 1)
            x = x+gamma*g
        elif step == 'ls':
            x, gamma = segment_search(f, grad_f, x, x+g)
        else:
            gamma = min(-np.dot(g, grad_f_x)/np.dot(g, (np.dot(step, g))), 1)
            x = x+gamma*g
        
        end = process_time()
        values.append(f(x))
        times.append(end-start+t1-t2)
        oracles.append(num_oracles)
        f_improv = f_old-f(x)
        
    return x, values, times, oracles, gaps

In [ ]:
m, n, sigma = 200, 500, 0.05

x_star = np.random.randn(n)
A = np.random.randn(m, n)
y = np.dot(A, x_star)+sigma*np.random.randn(m)
ATy = np.dot(A.transpose(), y)
f = lambda z: np.linalg.norm(y-np.dot(A, z[:n]-z[n:]), 2)**2
grad_f = grad(f)

def stoch_f(z, sample_indices):
    global A, y
    N = A.shape[0]
    x = z[:n] - z[n:]
    return np.linalg.norm(y[sample_indices] - A[sample_indices]@(z[:n]-z[n:]), ord=2)**2


# Compute the gradient of the sampled function with respect to x
stoch_grad_f = grad(stoch_f, 0)

L = 0.5 # not used

tau = np.linalg.norm(x_star, 1)
V = tau*np.identity(2*n)
x = V[np.random.randint(len(V))].copy()

lmo = lambda g: V[np.argmin(g)]

In [ ]:
f_tol, time_tol, align_tol = 1e-6, 50, 1e-3

rerun_deterministic = 0